# Cyber Threat Forecasting: Graph Model Verification

## Environment & Path Setup
Imports the necessary standard libraries and appends the project root to the system path, ensuring Python can locate the projects custom modules regardless of the notebook's location.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm  # Progress bar for notebooks
import json # for load the model config file

# --- Add Project Root to Path ---
# Get the directory where this notebook is running
current_dir = os.getcwd()

# Assume the project root is one level up
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path if not already there
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project Root added to path: {project_root}")

# Import your custom modules
from Transformer_Pipeline.Preprocessing.Load_Data import load_cyber_threat_data
from Transformer_Pipeline.Preprocessing.Cyber_Trend_to_Graph import (
    split_data, define_graph_structure, normalise_data, 
    create_sliding_windows, compute_dtw_matrix, 
    compute_shortest_path_matrices, compute_cluster_keys, 
    apply_double_exponential_smoothing
)
from Transformer_Pipeline.Graph_Dataset import CyberThreatGraphDataset
from Transformer_Pipeline.Models.PDFormer_Wrapper import PDFormerModel
from Transformers.Graph_Transformer.libcity.data.batch import Batch

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Setup Complete. Running on: {device}")

## Configuration & Constants

Defines the experiments hyperparameters, and sets the input window to 10 months and the forecast horizon to 36 months to align with the B-MTGNN benchmark.

In [ ]:
# --- B-MTGNN Benchmark Settings ---
CONSTANTS = {
    'TRAIN_SPLIT': 0.43,
    'VAL_SPLIT': 0.30,
    'WINDOW_SIZE': 10,       # 10 Months History
    'FORECAST_HORIZON': 36,  # 3 Years Future
    'BATCH_SIZE': 16,        # Small batch for small data
    'CORRELATION_THRESHOLD': 0.7,
    'N_CLUSTERS': 16,
    'OUTPUT_DIR': os.path.join(project_root, 'Processed_Data', 'graph_notebook_test')
}

# Ensure output directory exists
if not os.path.exists(CONSTANTS['OUTPUT_DIR']):
    os.makedirs(CONSTANTS['OUTPUT_DIR'])
    
print("Experimental Settings Locked:")
print(f" - Input Window: {CONSTANTS['WINDOW_SIZE']} Months")
print(f" - Forecast Horizon: {CONSTANTS['FORECAST_HORIZON']} Months")

## Data Ingestion & Verification

Loads the raw dataset using the corrected date parser to handle the 'MMM-YY' format, and verifies that the time-series index is sorted chronologically.

In [ ]:
raw_data_path = os.path.join(project_root, 'Data_Preparation', 'Cyber_Trend_Forecasting_All.csv')

# Load Data using fixed loader
df_raw = load_cyber_threat_data(raw_data_path)

if df_raw is not None:
    print("\n--- Data Verification ---")
    print(f"Index is monotonic increasing? {df_raw.index.is_monotonic_increasing}")
    print(f"Start Date: {df_raw.index.min().date()}")
    print(f"End Date:   {df_raw.index.max().date()}")
    
    # Visual Check
    display(df_raw.head(3))
else:
    print("Failed to load data.")

## Smoothing (Visual Check)

Applies Double Exponential Smoothing (DES) to the raw data and plots the result against the original values to visualise the noise reduction process.

In [ ]:
# Apply Double Exponential Smoothing
df_smooth = apply_double_exponential_smoothing(df_raw, alpha=0.1, beta=0.1)

# Plotting a specific threat to see the effect
target_col = 'Phishing-US' # Adjust based on checking columns
if target_col in df_raw.columns:
    plt.figure(figsize=(12, 5))
    plt.plot(df_raw.index, df_raw[target_col], label='Raw', alpha=0.5, color='gray')
    plt.plot(df_smooth.index, df_smooth[target_col], label='Smoothed (DES)', color='blue', linewidth=2)
    plt.title(f"Effect of Double Exponential Smoothing on {target_col}")
    plt.legend()
    plt.show()

## Graph Data Generation

Generate the Adjacency, DTW, and Shortest Path matrices required by the PDFormer architecture.



In [ ]:
print("--- Generating Graph Artifacts ---")

# 1. Split (using smoothed data)
train_df, val_df, test_df = split_data(df_smooth, CONSTANTS['TRAIN_SPLIT'], CONSTANTS['VAL_SPLIT'])

# 2. Adjacency Matrix
adj_matrix = define_graph_structure(train_df, CONSTANTS['CORRELATION_THRESHOLD'], CONSTANTS['OUTPUT_DIR'])

# 3. PDFormer Specifics (DTW, Shortest Path)
dtw_matrix = compute_dtw_matrix(train_df, CONSTANTS['OUTPUT_DIR'])
sh_mx, sd_mx = compute_shortest_path_matrices(adj_matrix, CONSTANTS['OUTPUT_DIR'])

print("Graph matrices generated.")

## Tensor Construction (Windowing)

Normalises the data and slices it into 4D tensors, checks the input and target shapes match the required dimensions (10-month history, 36-month forecast).

In [ ]:
# 1. Normalise
train_scaled, val_scaled, test_scaled, scaler = normalise_data(train_df, val_df, test_df, CONSTANTS['OUTPUT_DIR'])

# 2. Windowing
X_train, y_train = create_sliding_windows(train_scaled, CONSTANTS['WINDOW_SIZE'], CONSTANTS['FORECAST_HORIZON'])
X_val, y_val = create_sliding_windows(val_scaled, CONSTANTS['WINDOW_SIZE'], CONSTANTS['FORECAST_HORIZON'])
X_test, y_test = create_sliding_windows(test_scaled, CONSTANTS['WINDOW_SIZE'], CONSTANTS['FORECAST_HORIZON'])

# Save to temp .npz so the Dataset class can load it
np.savez(os.path.join(CONSTANTS['OUTPUT_DIR'], 'train.npz'), x=X_train, y=y_train)
np.savez(os.path.join(CONSTANTS['OUTPUT_DIR'], 'val.npz'), x=X_val, y=y_val)
np.savez(os.path.join(CONSTANTS['OUTPUT_DIR'], 'test.npz'), x=X_test, y=y_test)

# Cluster Keys (Must be done after windowing)
pattern_keys = compute_cluster_keys(X_train, CONSTANTS['N_CLUSTERS'], CONSTANTS['OUTPUT_DIR'])

print(f"\nTensor Shapes Check:")
print(f"X (Input):  {X_train.shape} -> (Samples, {CONSTANTS['WINDOW_SIZE']}, Nodes, 1)")
print(f"y (Target): {y_train.shape} -> (Samples, {CONSTANTS['FORECAST_HORIZON']}, Nodes, 1)")

## Dataset & Collator Setup

Initialises the `CyberThreatGraphDataset` and creates the PyTorch `DataLoader`, uses a custom collator function to batch data.

In [ ]:
# Custom Collator (from Train script)
def pdformer_collate_fn(batch_list):
    batch = Batch({'X': 'float', 'y': 'float'})
    for x_sample, y_sample in batch_list:
        batch.append((x_sample, y_sample))
    return batch

# Initialise Dataset
train_dataset = CyberThreatGraphDataset(data_dir=CONSTANTS['OUTPUT_DIR'], split='train')
val_dataset = CyberThreatGraphDataset(data_dir=CONSTANTS['OUTPUT_DIR'], split='val')

# Initialise Loader
train_loader = DataLoader(train_dataset, batch_size=CONSTANTS['BATCH_SIZE'], shuffle=True, collate_fn=pdformer_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=CONSTANTS['BATCH_SIZE'], shuffle=False, collate_fn=pdformer_collate_fn)

print("DataLoaders ready")

## Model Initialisation

Loads the model configuration and initialises the PDFormerModel.

In [ ]:
# Load Base Config
config_path = os.path.join(project_root, 'Transformer_Pipeline', 'pdformer_config.json')
with open(config_path, 'r') as f:
    config = json.load(f)

# OVERWRITE with experimental settings to be safe
config['input_window'] = CONSTANTS['WINDOW_SIZE']
config['output_window'] = CONSTANTS['FORECAST_HORIZON']
config['device'] = str(device)

# Load Static Features
static_features = train_dataset.get_static_features()
for k, v in static_features.items():
    if isinstance(v, np.ndarray):
        static_features[k] = torch.from_numpy(v).float().to(device)

# Auto-detect dimensions
sample_x, sample_y = train_dataset[0]
config['num_nodes'] = sample_x.shape[1]
config['feature_dim'] = sample_x.shape[2]
config['output_dim'] = sample_y.shape[2]

# Initialise Model
model = PDFormerModel(model_config=config, data_feature=static_features).to(device)
print(f"PDFormer Initialised. Input: {config['input_window']}, Output: {config['output_window']}")

## Training Loop

Runs a short training session within the notebook to demonstrate the optimisation process and calculate the validation metrics (RSE and RAE) in real-time.

In [ ]:
# Metrics Function (Global RSE/RAE)
def compute_metrics(all_preds, all_targets):
    preds_flat = all_preds.view(-1)
    targets_flat = all_targets.view(-1)
    target_mean = torch.mean(targets_flat)
    
    numerator_rse = torch.sqrt(torch.sum((preds_flat - targets_flat) ** 2))
    denominator_rse = torch.sqrt(torch.sum((targets_flat - target_mean) ** 2))
    rse = numerator_rse / (denominator_rse + 1e-7)
    
    numerator_rae = torch.sum(torch.abs(preds_flat - targets_flat))
    denominator_rae = torch.sum(torch.abs(targets_flat - target_mean))
    rae = numerator_rae / (denominator_rae + 1e-7)
    
    return rse.item(), rae.item()

optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'])
loss_fn = nn.L1Loss()

epochs = 5  # Short run for vignette
print("Starting Training...")

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
        batch.to_tensor(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = loss_fn(pred, batch['y'])
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    # Validation
    model.eval()
    preds_list, targets_list = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch.to_tensor(device)
            preds_list.append(model(batch).cpu())
            targets_list.append(batch['y'].cpu())
            
    rse, rae = compute_metrics(torch.cat(preds_list), torch.cat(targets_list))
    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val RSE: {rse:.4f} | Val RAE: {rae:.4f}")

## Forecast Plotting

Visualises the model's performance on a sample from the test set, plots the 10-month history, the 36-month ground truth, and the model's prediction on a single graph.

In [ ]:
# Visualise the first sample in the validation set
model.eval()
with torch.no_grad():
    batch = next(iter(val_loader))
    batch.to_tensor(device)
    prediction = model(batch)

# Extract data for Node 0 (e.g., Phishing)
# Shapes: (Batch, Time, Nodes, Feats)
history = batch['X'][0, :, 0, 0].cpu().numpy()
ground_truth = batch['y'][0, :, 0, 0].cpu().numpy()
forecast = prediction[0, :, 0, 0].cpu().numpy()

# Create time axes
t_history = np.arange(0, 10)
t_future = np.arange(10, 10 + 36)

plt.figure(figsize=(10, 5))
plt.plot(t_history, history, label='History (10 Months)', marker='o', color='black')
plt.plot(t_future, ground_truth, label='Ground Truth (36 Months)', alpha=0.5, color='gray')
plt.plot(t_future, forecast, label='PDFormer Forecast', linestyle='--', marker='x', color='blue')
plt.axvline(x=9.5, color='red', linestyle=':', label='Forecast Start')
plt.title("PDFormer Forecast: Node 0")
plt.legend()
plt.show()